# Dipole Angular Separation — Selected diaObjects (src-based)
Data source: `data_DIPOLES_03b/src_per_object/{oid}_src.parquet`
Sections 1–20: per-object panels. Sections 21–22: combined COSMOS figures.
- Sylvie Dagoret-Campagne — IJCLab/IN2P3/CNRS
- creation date : 2026-06-02
- last update : 2026-06-03

## 1. Imports & configuration

In [ ]:
import os, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy import stats
from scipy.integrate import simpson
from astropy.time import Time
from astropy.coordinates import EarthLocation, SkyCoord, AltAz
import astropy.units as u
from speclite import filters
import ref_index
from IPython.display import display

warnings.filterwarnings("ignore")
print(f"pandas {pd.__version__} | numpy {np.__version__}")

In [ ]:
try:
    import ipympl

    %matplotlib widget
    print("ipympl → %matplotlib widget")
except ImportError:
    %matplotlib inline
    print("→ %matplotlib inline")

In [ ]:
DIR_SRC_PER_OBJ = os.path.join("data_DIPOLES_03b", "src_per_object")
FILE_TOPRANKED = os.path.join("data_DIPOLES_03b", "topranked_objects_dipoles.csv")
NB_TAG = "DIPOLES_08b"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"src per obj : {os.path.abspath(DIR_SRC_PER_OBJ)}")
print(f"Figures     : {os.path.abspath(DIR_FIGS)}")

RUBIN_LAT_DEG = -30.244728
RUBIN_LON_DEG = -70.749417
RUBIN_HEIGHT_M = 2647.0
RUBIN_LOCATION = EarthLocation(
    lat=RUBIN_LAT_DEG * u.deg, lon=RUBIN_LON_DEG * u.deg, height=RUBIN_HEIGHT_M * u.m
)

BAND_COLORS = {"u": "#9b59b6", "g": "#2ecc71", "r": "#e74c3c", "i": "#e67e22", "z": "#3498db", "y": "#795548"}
BAND_ORDER = list("ugrizy")
LAM = np.linspace(0.3, 1.1, 3000)  # µm
RAD_TO_ARCSEC = 180.0 / np.pi * 3600.0
SEPCUT = 1.0
NCOLS = 3

plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name):
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  → saved {name}.{{pdf,png}}")


print("Configuration done.")

## 2. User selection — diaObjectId list
Set `SELECTED_DIAOBJ_IDS` or leave `None` to use `topranked_objects_dipoles.csv`.

In [ ]:
list_objsid = {
    0: 313985344866353157,  # rank 2  COSMOS, small dipoles, many bands
    1: 313853517840777344,  # rank 3  COSMOS
    2: 313972182542712999,  # rank 4  COSMOS, high and low dipoles
    3: 313871013109563545,  # rank 5  COSMOS
    4: 313871013420466334,  # rank 6  COSMOS
    5: 313998569477505082,  # rank 5  COSMOS
    6: 313994141002367046,  # rank 6  COSMOS, QSO
    7: 313888627167330394,
}

# ── USER SELECTION ─────────────────────────────────────────────────────────
SELECTED_DIAOBJ_IDS = [
    313888627167330394,
    313985344866353157,
    313853517840777344,
    313972182542712999,
    313871013109563545,
    313871013420466334,
    313998569477505082,
    313994141002367046,
]
MAX_OBJECTS = 9  # None = no limit
# ───────────────────────────────────────────────────────────────────────────

if SELECTED_DIAOBJ_IDS is None:
    df_top = pd.read_csv(FILE_TOPRANKED)
    SELECTED_DIAOBJ_IDS = df_top["diaObjectId"].tolist()
    display(df_top[["diaObjectId", "field", "n_dipoles", "dipole_fraction"]].head(20))

if MAX_OBJECTS is not None:
    SELECTED_DIAOBJ_IDS = SELECTED_DIAOBJ_IDS[:MAX_OBJECTS]

print(f"Final selection: {len(SELECTED_DIAOBJ_IDS)} diaObjectIds")
for oid in SELECTED_DIAOBJ_IDS:
    print(f"  {oid}")

## 3. Load per-object src parquets

In [ ]:
def parse_dipole_bool(series):
    def _cast(v):
        if isinstance(v, (bool, np.bool_)):
            return bool(v)
        if isinstance(v, (int, float)):
            return bool(v)
        if isinstance(v, str):
            return v.strip().lower() in ("true", "1", "yes")
        return False

    return series.apply(_cast)


frames, not_found = [], []
for oid in SELECTED_DIAOBJ_IDS:
    fpath = os.path.join(DIR_SRC_PER_OBJ, f"{oid}_src.parquet")
    if not os.path.exists(fpath):
        print(f"[MISSING] {oid}")
        not_found.append(oid)
        continue
    df = pd.read_parquet(fpath)
    df["is_dipole"] = (
        parse_dipole_bool(df["r:isDipole"].fillna(False)) if "r:isDipole" in df.columns else False
    )
    for col in (
        "r:midpointMjdTai",
        "r:ra",
        "r:dec",
        "r:dipoleAngle",
        "r:dipoleLength",
        "r:dipoleChi2",
        "r:dipoleFluxDiff",
        "r:psfFlux",
        "r:psfFluxErr",
    ):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    if "diaObjectId" not in df.columns:
        df["diaObjectId"] = oid
    n_all = len(df)
    n_dip = int(df["is_dipole"].sum())
    print(f"  {oid}  {n_all:5d} src  {n_dip:5d} dipoles  ({100 * n_dip / n_all:.1f}%)")
    frames.append(df)

if not frames:
    raise RuntimeError("No data loaded.")
df_raw = pd.concat(frames, ignore_index=True)
print(f"Total rows: {len(df_raw):,}")

In [ ]:
df_dip = df_raw[df_raw["is_dipole"]].copy().reset_index(drop=True)
print(f"Dipole sources: {len(df_dip):,} / {len(df_raw):,}")
display(
    df_dip.groupby("diaObjectId")[["r:band", "r:dipoleLength"]]
    .agg({"r:band": "count", "r:dipoleLength": "median"})
    .rename(columns={"r:band": "n_dip", "r:dipoleLength": "med_sep"})
)

## 4. DCR theory helpers

In [ ]:
def n_ciddor(lam, t=20.0, p=101325.0, rh=20.0):
    return ref_index.ciddor(wave=np.asarray(lam) * 1000.0, t=t, p=p, rh=rh)


N_LAM = n_ciddor(LAM)


def compute_sigma_n(band, lam=LAM, n_lam=N_LAM):
    bp = LSST_FILTERS[f"lsst2023-{band}"]
    lam_bp = bp.wavelength * 1e-4
    T = np.interp(lam, lam_bp, bp.response, left=0.0, right=0.0)
    w_raw = T / lam
    norm = simpson(w_raw, lam)
    if norm == 0.0:
        return 0.0
    w = w_raw / norm
    n_mean = simpson(w * n_lam, lam)
    return np.sqrt(simpson(w * (n_lam - n_mean) ** 2, lam))


print("compute_sigma_n() defined (needs LSST_FILTERS – run sec 5 first)")

In [ ]:
def compute_observing_geometry(ra_deg, dec_deg, mjd, location=RUBIN_LOCATION, batch_size=500):
    ra = np.asarray(ra_deg, float)
    dec = np.asarray(dec_deg, float)
    t = np.asarray(mjd, float)
    n = len(ra)
    para = np.full(n, np.nan)
    H_hr = np.full(n, np.nan)
    az = np.full(n, np.nan)
    alt = np.full(n, np.nan)
    za = np.full(n, np.nan)
    for i0 in range(0, n, batch_size):
        sl = slice(i0, min(i0 + batch_size, n))
        try:
            times = Time(t[sl], format="mjd", scale="tai").ut1
            coords = SkyCoord(ra=ra[sl] * u.deg, dec=dec[sl] * u.deg)
            lst = times.sidereal_time("apparent", longitude=location.lon)
            H_wrap = (lst - coords.ra).wrap_at(180 * u.deg)
            H_rad = H_wrap.to(u.rad).value
            H_hr[sl] = H_wrap.to(u.hourangle).value
            phi = location.lat.to(u.rad).value
            dec_rad = coords.dec.to(u.rad).value
            para[sl] = np.degrees(
                np.arctan2(np.sin(H_rad), np.tan(phi) * np.cos(dec_rad) - np.sin(dec_rad) * np.cos(H_rad))
            )
            frame = AltAz(obstime=times, location=location)
            altaz = coords.transform_to(frame)
            alt[sl] = altaz.alt.deg
            az[sl] = altaz.az.deg
            za[sl] = 90.0 - altaz.alt.deg
        except Exception as exc:
            print(f"  [warn] batch {i0}: {exc}")
    with np.errstate(divide="ignore", invalid="ignore"):
        airmass = np.where(za < 89.0, 1.0 / np.cos(np.radians(za)), np.nan)
    return pd.DataFrame(
        {
            "parallactic_angle_deg": para,
            "hour_angle_hr": H_hr,
            "hour_angle_deg": H_hr * 15.0,
            "azimuth_deg": az,
            "altitude_deg": alt,
            "zenith_angle_deg": za,
            "sin_zenith": np.sin(np.radians(za)),
            "tan_zenith": np.tan(np.radians(za)),
            "airmass": airmass,
        }
    )


def theory_parallactic(H_deg, ra_deg, dec_deg):
    H = np.deg2rad(H_deg)
    phi = np.deg2rad(RUBIN_LAT_DEG)
    dec = np.deg2rad(dec_deg)
    return np.degrees(np.arctan2(np.sin(H), np.tan(phi) * np.cos(dec) - np.sin(dec) * np.cos(H)))


def theory_sinz(H_deg, ra_deg, dec_deg):
    H = np.deg2rad(H_deg)
    phi = np.deg2rad(RUBIN_LAT_DEG)
    dec = np.deg2rad(dec_deg)
    cosz = np.sin(phi) * np.sin(dec) + np.cos(phi) * np.cos(dec) * np.cos(H)
    return np.sqrt(np.maximum(0.0, 1.0 - cosz**2))


print("Geometry helpers defined.")

## 5. Load LSST filters & pre-compute sigma_n

In [ ]:
_lsst_seq = filters.load_filters("lsst2023-*")
LSST_FILTERS = {f.name: f for f in _lsst_seq}
SIGMA_N = {b: compute_sigma_n(b) for b in BAND_ORDER}
print("sigma_n per band:")
for b, sn in SIGMA_N.items():
    print(f"  {b}:  {sn:.5e}   l_dip(z=45°) = {sn * np.tan(np.deg2rad(45)) * RAD_TO_ARCSEC:.5f} arcsec")

## 6. Compute observing geometry

In [ ]:
need = ["r:ra", "r:dec", "r:midpointMjdTai"]
mask_valid = df_dip[need].notna().all(axis=1)
df_clean = df_dip[mask_valid].copy().reset_index(drop=True)
print(f"Dipole rows with complete coords: {len(df_clean):,} (dropped {(~mask_valid).sum()})")

print("Computing observing geometry …", end=" ", flush=True)
geo = compute_observing_geometry(
    df_clean["r:ra"].values, df_clean["r:dec"].values, df_clean["r:midpointMjdTai"].values
)
df_clean = pd.concat([df_clean, geo], axis=1)
print("done.")

if "r:dipoleAngle" in df_clean.columns:
    diff = (
        df_clean["r:dipoleAngle"].values - df_clean["parallactic_angle_deg"].values + 180.0
    ) % 360.0 - 180.0
    df_clean["delta_dipole_para"] = diff
    adiff = np.abs(diff)
    df_clean["delta_dipole_para_folded"] = np.where(adiff <= 90.0, adiff, 180.0 - adiff)

if "r:band" in df_clean.columns:
    df_clean["dipole_length_pred"] = df_clean["r:band"].map(SIGMA_N) * df_clean["tan_zenith"] * RAD_TO_ARCSEC

print(f"df_clean shape: {df_clean.shape}")
display(
    df_clean[
        [
            "diaObjectId",
            "r:band",
            "r:dipoleAngle",
            "r:dipoleLength",
            "parallactic_angle_deg",
            "hour_angle_hr",
            "zenith_angle_deg",
            "airmass",
        ]
    ].describe()
)

## 7. Build per-object metadata

In [ ]:
df_meta = (
    pd.read_csv(FILE_TOPRANKED).set_index("diaObjectId") if os.path.exists(FILE_TOPRANKED) else pd.DataFrame()
)
OBJECT_INFO = {}
for oid in SELECTED_DIAOBJ_IDS:
    sub = df_clean[df_clean["diaObjectId"] == oid]
    if sub.empty:
        continue
    field = df_meta.loc[oid, "field"] if (oid in df_meta.index and "field" in df_meta.columns) else "?"
    n_dip = len(sub)
    OBJECT_INFO[oid] = {
        "label": f"{oid}\n({field}, n={n_dip})",
        "ra": sub["r:ra"].mean(),
        "dec": sub["r:dec"].mean(),
        "field": field,
        "n_dip": n_dip,
    }
OBJECT_IDS_PRESENT = list(OBJECT_INFO.keys())
print(f"Objects with dipole data: {len(OBJECT_IDS_PRESENT)}")
for oid, info in OBJECT_INFO.items():
    print(f"  {oid}  field={info['field']}  RA={info['ra']:.4f}  Dec={info['dec']:.4f}  n={info['n_dip']}")

## 8. Plot helpers

In [ ]:
def _grid_shape(n):
    return math.ceil(n / NCOLS), NCOLS


def _band_legend_elements():
    e = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    e.append(plt.Line2D([0], [0], ls="--", color="black", lw=0.8, alpha=0.6, label="uniform"))
    return e


def profile_median(x, y, bins, min_count=1):
    x = np.asarray(x).ravel()
    y = np.asarray(y).ravel()
    bins = np.asarray(bins)
    nbins = len(bins) - 1
    idx = np.digitize(x, bins) - 1
    y_med = np.full(nbins, np.nan)
    y_rms = np.full(nbins, np.nan)
    y_err = np.full(nbins, np.nan)
    for i in range(nbins):
        yi = y[idx == i]
        if yi.size >= min_count:
            med = np.median(yi)
            rms = 1.4826 * np.median(np.abs(yi - med))
            y_med[i] = med
            y_rms[i] = rms
            y_err[i] = rms / np.sqrt(yi.size)
    return 0.5 * (bins[:-1] + bins[1:]), y_med, y_err, y_rms


print("Plot helpers defined.")

In [ ]:
def rose_stacked_bands(ax, df_obj, angle_col, n_bins=36, title="", show_uniform=True, angle_range=(0, 360)):
    lo, hi = angle_range
    span = hi - lo
    bin_edges = np.linspace(lo, hi, n_bins + 1)
    centers = np.radians((bin_edges[:-1] + bin_edges[1:]) / 2.0)
    width = 2 * np.pi * (span / 360.0) / n_bins * 0.90
    bands_present = (
        [b for b in BAND_ORDER if b in df_obj["r:band"].dropna().unique()]
        if "r:band" in df_obj.columns
        else []
    )
    bottom = np.zeros(n_bins)
    for band in bands_present:
        vals = df_obj.loc[df_obj["r:band"] == band, angle_col].dropna().values
        vals = lo + (vals - lo) % span
        cnts, _ = np.histogram(vals, bins=bin_edges)
        ax.bar(
            centers,
            cnts,
            width=width,
            bottom=bottom,
            color=BAND_COLORS.get(band, "grey"),
            alpha=0.85,
            edgecolor="white",
            linewidth=0.3,
            label=band,
        )
        bottom += cnts
    if show_uniform and bottom.sum() > 0:
        u_val = bottom.sum() / n_bins
        ax.plot(
            np.linspace(np.radians(lo), np.radians(hi), 360),
            np.full(360, u_val),
            "--",
            color="black",
            lw=0.8,
            alpha=0.6,
            label="uniform",
        )
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ax.set_thetalim(np.radians(lo), np.radians(hi))
    ax.tick_params(labelsize=7)
    ax.set_title(title, va="bottom", pad=14, fontsize=7)


def figure_rose_per_object(df, angle_col, suptitle, figname, n_bins=36, xlabel="", angle_range=(0, 360)):
    nrows, ncols = _grid_shape(len(OBJECT_IDS_PRESENT))
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(ncols * 4.0, nrows * 4.2),
        subplot_kw={"projection": "polar"},
        layout="constrained",
    )
    axes_flat = np.array(axes).ravel()
    for idx, oid in enumerate(OBJECT_IDS_PRESENT):
        ax = axes_flat[idx]
        sub = df[df["diaObjectId"] == oid]
        if sub.empty or angle_col not in sub.columns:
            ax.set_visible(False)
            continue
        rose_stacked_bands(
            ax,
            sub,
            angle_col,
            n_bins=n_bins,
            title=OBJECT_INFO[oid]["label"] + f"\nn={sub[angle_col].notna().sum():,}",
            angle_range=angle_range,
        )
    for idx in range(len(OBJECT_IDS_PRESENT), nrows * ncols):
        axes_flat[idx].set_visible(False)
    fig.suptitle(suptitle + (f"  [{xlabel}]" if xlabel else ""), y=1.01, fontsize=11)
    fig.legend(
        handles=_band_legend_elements(),
        loc="lower center",
        ncol=8,
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.03),
    )
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("rose helpers defined.")

In [ ]:
def figure_scatter_profile_per_object(
    df, x_col, y_col, suptitle, figname, xlabel="", ylabel="", n_bins=15, xlim=None, ylim=None, abs_x=False
):
    nrows, ncols = _grid_shape(len(OBJECT_IDS_PRESENT))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8), layout="constrained")
    axes_flat = np.array(axes).ravel()

    def _draw(ax, sub, title):
        if sub.empty:
            ax.set_visible(False)
            return
        sub = sub[sub[x_col].notna() & sub[y_col].notna()]
        if len(sub) < 5:
            ax.set_visible(False)
            return
        xv = np.abs(sub[x_col].values) if abs_x else sub[x_col].values
        yv = sub[y_col].values
        if "r:band" in sub.columns:
            for band in [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]:
                ib = sub["r:band"] == band
                xb = np.abs(sub.loc[ib, x_col].values) if abs_x else sub.loc[ib, x_col].values
                ax.scatter(
                    xb,
                    sub.loc[ib, y_col].values,
                    s=10,
                    alpha=0.5,
                    color=BAND_COLORS.get(band, "grey"),
                    rasterized=True,
                    label=band,
                )
        else:
            ax.scatter(xv, yv, s=10, alpha=0.5, color="steelblue", rasterized=True)
        x_min = xlim[0] if xlim else xv.min()
        x_max = xlim[1] if xlim else xv.max()
        bed = np.linspace(x_min, x_max, n_bins + 1)
        xc, med, mad = [], [], []
        for j in range(n_bins):
            sel = (xv >= bed[j]) & (xv < bed[j + 1])
            if sel.sum() < 3:
                continue
            ybin = yv[sel]
            m = np.median(ybin)
            xc.append((bed[j] + bed[j + 1]) / 2)
            med.append(m)
            mad.append(np.median(np.abs(ybin - m)))
        if xc:
            xc = np.array(xc)
            med = np.array(med)
            mad = np.array(mad)
            ax.plot(xc, med, "k-", lw=1.5, label="median")
            ax.fill_between(xc, med - mad, med + mad, color="black", alpha=0.15, label="±MAD")
        r_s, _ = stats.spearmanr(xv, yv)
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f"{title}  ρ={r_s:.3f}", fontsize=7)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        ax.tick_params(labelsize=7)

    for idx, oid in enumerate(OBJECT_IDS_PRESENT):
        _draw(axes_flat[idx], df[df["diaObjectId"] == oid], OBJECT_INFO[oid]["label"])
    for idx in range(len(OBJECT_IDS_PRESENT), nrows * ncols):
        axes_flat[idx].set_visible(False)
    le = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    le += [
        plt.Line2D([0], [0], color="black", lw=1.5, label="median"),
        mpatches.Patch(facecolor="black", alpha=0.15, label="±MAD"),
    ]
    fig.legend(
        handles=le, loc="lower center", ncol=len(le), fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.04)
    )
    fig.suptitle(suptitle, y=1.00, fontsize=10)
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("figure_scatter_profile_per_object() defined.")

In [ ]:
def figure_profilehist_with_theory_per_object(
    df,
    x_col,
    y_col,
    x_col_bins,
    theory_func,
    suptitle,
    figname,
    xlabel="",
    ylabel="",
    xlim=(-180, 180),
    ylim=None,
    x_is_hours=False,
):
    nrows, ncols = _grid_shape(len(OBJECT_IDS_PRESENT))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.8), layout="constrained")
    axes_flat = np.array(axes).ravel()
    H_th = np.linspace(-180, 180, 720)

    def _draw(ax, sub, oid):
        if sub.empty:
            ax.set_visible(False)
            return
        sub = sub[sub[x_col].notna() & sub[y_col].notna()]
        if len(sub) < 3:
            ax.set_visible(False)
            return
        ra = OBJECT_INFO[oid]["ra"]
        dec = OBJECT_INFO[oid]["dec"]
        if "r:band" in sub.columns:
            for band in [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()]:
                ib = sub["r:band"] == band
                xv = np.asarray(sub.loc[ib, x_col]).ravel()
                yv = np.asarray(sub.loc[ib, y_col]).ravel()
                xc, ym, ye, _ = profile_median(xv, yv, x_col_bins)
                ax.errorbar(
                    xc, ym, yerr=ye, fmt="o", capsize=3, color=BAND_COLORS.get(band, "grey"), ms=4, label=band
                )
        else:
            xv = np.asarray(sub[x_col]).ravel()
            yv = np.asarray(sub[y_col]).ravel()
            xc, ym, ye, _ = profile_median(xv, yv, x_col_bins)
            ax.errorbar(xc, ym, yerr=ye, fmt="o", capsize=1, color="k")
        y_th = theory_func(H_th, ra, dec)
        x_th = H_th / 15.0 if x_is_hours else H_th
        ax.plot(x_th, y_th, "k-", lw=1.5, zorder=5, label="theory")
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(OBJECT_INFO[oid]["label"] + f"  (n={len(sub):,})", fontsize=7)
        if xlim:
            ax.set_xlim(xlim)
        if ylim:
            ax.set_ylim(ylim)
        ax.tick_params(labelsize=7)

    for idx, oid in enumerate(OBJECT_IDS_PRESENT):
        _draw(axes_flat[idx], df[df["diaObjectId"] == oid], oid)
    for idx in range(len(OBJECT_IDS_PRESENT), nrows * ncols):
        axes_flat[idx].set_visible(False)
    le = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]
    le.append(plt.Line2D([0], [0], color="black", lw=1.5, label="theory"))
    fig.legend(
        handles=le, loc="lower center", ncol=len(le), fontsize=8, frameon=False, bbox_to_anchor=(0.5, -0.04)
    )
    fig.suptitle(suptitle, y=1.00, fontsize=10)
    plt.tight_layout()
    savefig(figname)
    plt.show()


print("figure_profilehist_with_theory_per_object() defined.")

## 9. Summary statistics — `r:dipoleLength` per diaObject × band

In [ ]:
SEP_COL = "r:dipoleLength"
if SEP_COL in df_clean.columns:
    display(
        df_clean.groupby(["diaObjectId", "r:band"])[SEP_COL]
        .agg(
            n="count",
            min=np.min,
            p16=lambda x: np.percentile(x.dropna(), 16),
            median=np.median,
            p84=lambda x: np.percentile(x.dropna(), 84),
            max=np.max,
        )
        .round(4)
    )
else:
    print(f"{SEP_COL} not available.")

## 10. Log-binned `r:dipoleLength` histograms — per diaObject, stacked by band

In [ ]:
N_BINS = 40
if SEP_COL in df_clean.columns:
    vg = df_clean[SEP_COL].dropna().values
    vg = vg[vg > 0]
    lo_global, hi_global = vg.min(), vg.max()
    print(f"Global range: [{lo_global:.4f}, {hi_global:.4f}]")
else:
    lo_global, hi_global = 0.01, 10.0
log_edges = np.logspace(np.log10(lo_global), np.log10(hi_global), N_BINS + 1)
legend_handles = [mpatches.Patch(facecolor=BAND_COLORS[b], label=b) for b in BAND_ORDER if b in BAND_COLORS]

In [ ]:
nrows, ncols = _grid_shape(len(OBJECT_IDS_PRESENT))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.6), layout="constrained")
axes_flat = np.array(axes).ravel()
for idx, oid in enumerate(OBJECT_IDS_PRESENT):
    ax = axes_flat[idx]
    sub = df_clean[df_clean["diaObjectId"] == oid]
    if sub.empty or SEP_COL not in sub.columns:
        ax.set_visible(False)
        continue
    bands_present = (
        [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
    )
    bottom = np.zeros(N_BINS)
    for band in bands_present:
        vb = sub.loc[sub["r:band"] == band, SEP_COL].dropna().values
        vb = vb[vb > 0]
        cnts, _ = np.histogram(vb, bins=log_edges)
        ax.bar(
            log_edges[:-1],
            cnts,
            width=np.diff(log_edges),
            bottom=bottom,
            align="edge",
            color=BAND_COLORS.get(band, "grey"),
            alpha=0.85,
            edgecolor="white",
            linewidth=0.3,
            label=band,
        )
        bottom += cnts
    vf = sub[SEP_COL].dropna().values
    vf = vf[vf > 0]
    if len(vf) > 0:
        ax.axvline(np.median(vf), color="black", lw=1.2, ls="--", alpha=0.7, label=f"med={np.median(vf):.3f}")
        ax.legend(fontsize=6, loc="upper right", framealpha=0.5)
    ax.set_xscale("log")
    ax.set_xlabel(SEP_COL, fontsize=8)
    ax.set_ylabel("N dipoles", fontsize=8)
    ax.set_title(OBJECT_INFO[oid]["label"] + f"  n={len(vf):,}", fontsize=7)
    ax.set_xlim(lo_global, hi_global)
    ax.tick_params(labelsize=7)
for idx in range(len(OBJECT_IDS_PRESENT), nrows * ncols):
    axes_flat[idx].set_visible(False)
fig.legend(
    handles=legend_handles, loc="lower center", ncol=6, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.04)
)
fig.suptitle("Dipole angular separation — log-binned histograms per diaObject", y=1.01, fontsize=11)
savefig("hist_dipoleLength_logbin_per_object")
plt.show()

## 11. Normalised (density) histograms — per diaObject

In [ ]:
log_widths = np.diff(log_edges)
nrows, ncols = _grid_shape(len(OBJECT_IDS_PRESENT))
fig2, axes2 = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.6), layout="constrained")
axes2_flat = np.array(axes2).ravel()
for idx, oid in enumerate(OBJECT_IDS_PRESENT):
    ax = axes2_flat[idx]
    sub = df_clean[df_clean["diaObjectId"] == oid]
    if sub.empty or SEP_COL not in sub.columns:
        ax.set_visible(False)
        continue
    bands_present = (
        [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
    )
    n_tot = sub[SEP_COL].dropna().shape[0]
    bottom = np.zeros(N_BINS)
    for band in bands_present:
        vb = sub.loc[sub["r:band"] == band, SEP_COL].dropna().values
        vb = vb[vb > 0]
        cnts, _ = np.histogram(vb, bins=log_edges)
        density = cnts / (n_tot * log_widths) if n_tot > 0 else cnts
        ax.bar(
            log_edges[:-1],
            density,
            width=log_widths,
            bottom=bottom,
            align="edge",
            color=BAND_COLORS.get(band, "grey"),
            alpha=0.85,
            edgecolor="white",
            linewidth=0.3,
            label=band,
        )
        bottom += density
    vf = sub[SEP_COL].dropna().values
    vf = vf[vf > 0]
    if len(vf) > 0:
        ax.axvline(np.median(vf), color="black", lw=1.2, ls="--", alpha=0.7, label=f"med={np.median(vf):.3f}")
        ax.legend(fontsize=6, loc="upper right", framealpha=0.5)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(SEP_COL, fontsize=8)
    ax.set_ylabel("Probability density", fontsize=8)
    ax.set_title(OBJECT_INFO[oid]["label"] + f"  n={n_tot:,}", fontsize=7)
    ax.set_xlim(lo_global, hi_global)
    ax.tick_params(labelsize=7)
for idx in range(len(OBJECT_IDS_PRESENT), nrows * ncols):
    axes2_flat[idx].set_visible(False)
fig2.legend(
    handles=legend_handles, loc="lower center", ncol=6, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.04)
)
fig2.suptitle("Dipole angular separation — normalised log histograms per diaObject", y=1.01, fontsize=11)
savefig("hist_dipoleLength_logbin_per_object_normed")
plt.show()

## 12. Per-band overlay — normalised histograms per diaObject

In [ ]:
nrows, ncols = _grid_shape(len(OBJECT_IDS_PRESENT))
fig5, axes5 = plt.subplots(nrows, ncols, figsize=(ncols * 4.5, nrows * 3.6), layout="constrained")
axes5_flat = np.array(axes5).ravel()
for idx, oid in enumerate(OBJECT_IDS_PRESENT):
    ax = axes5_flat[idx]
    sub = df_clean[df_clean["diaObjectId"] == oid]
    if sub.empty or SEP_COL not in sub.columns:
        ax.set_visible(False)
        continue
    bands_present = (
        [b for b in BAND_ORDER if b in sub["r:band"].dropna().unique()] if "r:band" in sub.columns else []
    )
    n_tot = sub[SEP_COL].dropna().shape[0]
    for band in bands_present:
        vb = sub.loc[sub["r:band"] == band, SEP_COL].dropna().values
        vb = vb[vb > 0]
        if len(vb) < 3:
            continue
        cnts, _ = np.histogram(vb, bins=log_edges)
        density = cnts / (len(vb) * log_widths)
        centers = np.sqrt(log_edges[:-1] * log_edges[1:])
        ax.step(
            np.concatenate([[log_edges[0]], centers]),
            np.concatenate([[0], density]),
            where="pre",
            color=BAND_COLORS.get(band, "grey"),
            lw=1.5,
            label=band,
        )
        ax.axvline(np.median(vb), color=BAND_COLORS.get(band, "grey"), lw=0.8, ls=":", alpha=0.7)
    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlabel(SEP_COL, fontsize=8)
    ax.set_ylabel("Probability density", fontsize=8)
    ax.set_title(OBJECT_INFO[oid]["label"] + f"  n={n_tot:,}", fontsize=7)
    ax.set_xlim(lo_global, hi_global)
    ax.legend(fontsize=7, loc="upper right", framealpha=0.5)
    ax.tick_params(labelsize=7)
for idx in range(len(OBJECT_IDS_PRESENT), nrows * ncols):
    axes5_flat[idx].set_visible(False)
fig5.legend(
    handles=legend_handles, loc="lower center", ncol=6, fontsize=9, frameon=False, bbox_to_anchor=(0.5, -0.04)
)
fig5.suptitle(
    "Dipole separation — per-band normalised histograms per diaObject (dotted=band median)",
    y=1.01,
    fontsize=11,
)
savefig("hist_dipoleLength_perband_per_object")
plt.show()

## 13. Overlay — all selected diaObjects on a single panel

In [ ]:
colors_cycle = plt.rcParams["axes.prop_cycle"].by_key()["color"]
fig_ov, ax_ov = plt.subplots(figsize=(7, 4), layout="constrained")
for i, oid in enumerate(OBJECT_IDS_PRESENT):
    sub = df_clean[df_clean["diaObjectId"] == oid]
    vals = sub[SEP_COL].dropna().values
    vals = vals[vals > 0]
    if len(vals) < 5:
        continue
    cnts, _ = np.histogram(vals, bins=log_edges)
    density = cnts / (len(vals) * log_widths)
    centers = np.sqrt(log_edges[:-1] * log_edges[1:])
    color = colors_cycle[i % len(colors_cycle)]
    ax_ov.step(
        np.concatenate([[log_edges[0]], centers]),
        np.concatenate([[0], density]),
        where="pre",
        color=color,
        lw=1.5,
        label=f"{oid} ({OBJECT_INFO[oid]['field']})",
    )
    ax_ov.axvline(np.median(vals), color=color, lw=0.8, ls=":", alpha=0.8)
ax_ov.set_xscale("log")
ax_ov.set_yscale("log")
ax_ov.set_xlabel(f"{SEP_COL}  [log scale]", fontsize=9)
ax_ov.set_ylabel("Probability density", fontsize=9)
ax_ov.set_title("Dipole separation — overlay all selected diaObjects (dotted=median)", fontsize=10)
ax_ov.legend(fontsize=7, loc="upper right")
ax_ov.set_xlim(lo_global, hi_global)
savefig("hist_dipoleLength_overlay_objects")
plt.show()

## 14. Rose diagram — `r:dipoleAngle` per diaObject (all sources)

In [ ]:
figure_rose_per_object(
    df_clean,
    "r:dipoleAngle",
    suptitle="Rose diagram — r:dipoleAngle per diaObject",
    figname="rose_dipoleAngle_per_object",
    xlabel="CCW from East (pixel axis)",
) if not df_clean.empty else print("No data.")

## 15. Dipole length vs tan(z) — per diaObject

In [ ]:
if "tan_zenith" in df_clean.columns and SEP_COL in df_clean.columns:
    figure_scatter_profile_per_object(
        df_clean,
        "tan_zenith",
        SEP_COL,
        suptitle="r:dipoleLength vs tan(z) — per diaObject",
        figname="scatter_dipoleLength_vs_tanz_per_object",
        xlabel="tan z",
        ylabel="r:dipoleLength (arcsec)",
        n_bins=20,
        xlim=(0.0, 2.0),
        ylim=(0.0, 0.3),
    )
else:
    print("Missing column — skipping.")

## 16. Measured vs predicted dipole length — per diaObject

In [ ]:
if "dipole_length_pred" in df_clean.columns and SEP_COL in df_clean.columns:
    figure_scatter_profile_per_object(
        df_clean,
        "dipole_length_pred",
        SEP_COL,
        suptitle="r:dipoleLength vs DCR-predicted length — per diaObject",
        figname="scatter_dipoleLength_vs_prediction_per_object",
        xlabel="predicted (arcsec)",
        ylabel="r:dipoleLength (arcsec)",
        n_bins=20,
        xlim=(0.0, 0.5),
        ylim=(0.0, 0.3),
    )
else:
    print("dipole_length_pred missing — skipping.")

## 17. Apply separation cut  (`r:dipoleLength` < SEPCUT)

In [ ]:
df_selected = df_clean[df_clean[SEP_COL] < SEPCUT].copy()
print(f"After cut r:dipoleLength < {SEPCUT}: {len(df_selected):,} / {len(df_clean):,} sources")

In [ ]:
figure_rose_per_object(
    df_selected,
    "r:dipoleAngle",
    suptitle=f"Rose diagram — r:dipoleAngle  (sep < {SEPCUT})",
    figname="rose_dipoleAngle_per_object_selected",
    xlabel="CCW from East",
) if not df_selected.empty else print("No data after cut.")

## 18. Dipole angle vs Hour angle — profile + theory (per diaObject, all sources)

In [ ]:
if "parallactic_angle_deg" in df_clean.columns:
    figure_profilehist_with_theory_per_object(
        df_clean,
        "hour_angle_deg",
        "r:dipoleAngle",
        np.linspace(-120, 120, 100),
        theory_parallactic,
        suptitle="Dipole angle vs H (deg) — profile + theory",
        figname="profile_theory_parallactic_vs_Hdeg_per_object",
        xlabel="H (deg)",
        ylabel="Dipole angle (deg)",
        xlim=(-120, 120),
        ylim=(-180, 180),
    )

In [ ]:
if "parallactic_angle_deg" in df_clean.columns:
    figure_profilehist_with_theory_per_object(
        df_clean,
        "hour_angle_hr",
        "r:dipoleAngle",
        np.linspace(-6, 6, 100),
        theory_parallactic,
        suptitle="Dipole angle vs H (hour) — profile + theory",
        figname="profile_theory_parallactic_vs_Hhr_per_object",
        xlabel="H (hour)",
        ylabel="Dipole angle (deg)",
        xlim=(-6, 6),
        ylim=(-180, 180),
        x_is_hours=True,
    )

## 19. Same — after separation cut

In [ ]:
if not df_selected.empty and "parallactic_angle_deg" in df_selected.columns:
    figure_profilehist_with_theory_per_object(
        df_selected,
        "hour_angle_deg",
        "r:dipoleAngle",
        np.linspace(-120, 120, 51),
        theory_parallactic,
        suptitle=f"Dipole angle vs H (deg) — profile + theory  (sep < {SEPCUT})",
        figname="profile_theory_parallactic_vs_Hdeg_per_object_selected",
        xlabel="H (deg)",
        ylabel="Dipole angle (deg)",
        xlim=(-120, 120),
        ylim=(-180, 180),
    )

In [ ]:
if not df_selected.empty and "parallactic_angle_deg" in df_selected.columns:
    figure_profilehist_with_theory_per_object(
        df_selected,
        "hour_angle_hr",
        "r:dipoleAngle",
        np.linspace(-6, 6, 51),
        theory_parallactic,
        suptitle=f"Dipole angle vs H (hour) — profile + theory  (sep < {SEPCUT})",
        figname="profile_theory_parallactic_vs_Hhr_per_object_selected",
        xlabel="H (hour)",
        ylabel="Dipole angle (deg)",
        xlim=(-6, 6),
        ylim=(-180, 180),
        x_is_hours=True,
    )

## 21. Combined COSMOS figures — all selected diaObjects merged

All dipole sources from the selected objects are pooled into a single dataset.
Since all objects belong to the **COSMOS DDF**, a single theoretical curve
η(H) computed at the COSMOS field centre (RA = 150.1191°, Dec = +2.2058°)
serves as the reference.

Four figures (21a–21d) are produced, then repeated after the SEPCUT filter (22a–22d):

- 21a/22a — dipole angle vs H (deg) + COSMOS theory
- 21b/22b — dipole angle vs H (hour) + COSMOS theory
- 21c/22c — rose diagram `r:dipoleAngle`
- 21d/22d — dipole length vs tan(z) + DCR prediction lines per band

In [ ]:
COSMOS_RA_DEG = 150.1191
COSMOS_DEC_DEG = 2.2058
z_cosmos = abs(RUBIN_LAT_DEG - COSMOS_DEC_DEG)
print(f"COSMOS field centre: RA = {COSMOS_RA_DEG}°  Dec = {COSMOS_DEC_DEG}°")
print(f"Zenith distance at transit: {z_cosmos:.2f}°   sin(z) = {np.sin(np.radians(z_cosmos)):.4f}")
print(f"Combined dataset (dipole-flagged, all objects): {len(df_clean):,} sources")
if "r:band" in df_clean.columns:
    print(f"  bands present: {sorted(df_clean['r:band'].dropna().unique())}")

In [ ]:
def figure_combined_profile_theory(
    df,
    x_col,
    y_col,
    x_col_bins,
    field_ra,
    field_dec,
    suptitle,
    figname,
    xlabel="",
    ylabel="",
    xlim=(-120, 120),
    ylim=(-180, 180),
    x_is_hours=False,
    scatter_alpha=0.20,
    scatter_s=5,
):
    """
    Single-panel combined figure for COSMOS:
      - semi-transparent scatter coloured by band
      - per-band median profile with MAD error bars
      - single η(H) theory curve at (field_ra, field_dec)
      - Spearman ρ annotation
    """
    mask = df[x_col].notna() & df[y_col].notna()
    df_p = df[mask].copy()
    if df_p.empty:
        print(f"[combined] no valid data for {figname}")
        return

    fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
    H_th = np.linspace(-180, 180, 720)
    y_th = theory_parallactic(H_th, field_ra, field_dec)
    x_th = H_th / 15.0 if x_is_hours else H_th
    bands = (
        [b for b in BAND_ORDER if b in df_p["r:band"].dropna().unique()] if "r:band" in df_p.columns else []
    )

    for band in bands:
        ib = df_p["r:band"] == band
        ax.scatter(
            df_p.loc[ib, x_col].values,
            df_p.loc[ib, y_col].values,
            s=scatter_s,
            alpha=scatter_alpha,
            color=BAND_COLORS.get(band, "grey"),
            rasterized=True,
            zorder=1,
        )
    for band in bands:
        ib = df_p["r:band"] == band
        xv = df_p.loc[ib, x_col].values
        yv = df_p.loc[ib, y_col].values
        xc, ym, ye, _ = profile_median(xv, yv, x_col_bins, min_count=3)
        valid = np.isfinite(ym)
        if valid.sum() < 2:
            continue
        color = BAND_COLORS.get(band, "grey")
        ax.errorbar(
            xc[valid],
            ym[valid],
            yerr=ye[valid],
            fmt="o",
            capsize=3,
            color=color,
            ms=5,
            markeredgecolor="black",
            markeredgewidth=0.4,
            elinewidth=1.2,
            zorder=3,
            label=band,
        )

    ax.plot(x_th, y_th, "k-", lw=2.2, zorder=5, label=f"theory  COSMOS ({field_ra:.3f}°, {field_dec:+.3f}°)")

    r_s, p_s = stats.spearmanr(df_p[x_col].values, df_p[y_col].values)
    ax.text(
        0.02,
        0.97,
        f"Spearman ρ = {r_s:.3f}  (p = {p_s:.2e})   n = {len(df_p):,}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8),
    )

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=10)
    ax.set_title(suptitle, fontsize=11)
    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)
    ax.grid(True, alpha=0.3)
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, fontsize=8, loc="lower right", framealpha=0.8, ncol=2)
    savefig(figname)
    plt.show()


def figure_combined_rose(df, angle_col, suptitle, figname, n_bins=36, angle_range=(0, 360)):
    """Single polar rose — all selected objects merged, stacked by band."""
    fig, ax = plt.subplots(figsize=(5, 5), subplot_kw={"projection": "polar"}, layout="constrained")
    n_tot = df[angle_col].notna().sum() if angle_col in df.columns else 0
    rose_stacked_bands(
        ax,
        df,
        angle_col,
        n_bins=n_bins,
        title=f"COSMOS — all selected objects\nn = {n_tot:,}",
        angle_range=angle_range,
    )
    fig.suptitle(suptitle, y=1.03, fontsize=11)
    fig.legend(
        handles=_band_legend_elements(),
        loc="lower center",
        ncol=8,
        fontsize=8,
        frameon=False,
        bbox_to_anchor=(0.5, -0.06),
    )
    savefig(figname)
    plt.show()


print("figure_combined_profile_theory(), figure_combined_rose() defined.")

### 21a. Combined — dipole angle vs H (deg) + COSMOS theory

In [ ]:
if "parallactic_angle_deg" in df_clean.columns:
    figure_combined_profile_theory(
        df_clean,
        "hour_angle_deg",
        "r:dipoleAngle",
        np.linspace(-120, 120, 61),
        COSMOS_RA_DEG,
        COSMOS_DEC_DEG,
        suptitle="Combined COSMOS — dipole angle vs H (deg), all selected diaObjects",
        figname="combined_profile_theory_parallactic_vs_Hdeg_COSMOS",
        xlabel="Hour angle H (deg)",
        ylabel="r:dipoleAngle (deg, CCW from East)",
        xlim=(-120, 120),
        ylim=(-180, 180),
        x_is_hours=False,
    )
else:
    print("No geometry — skipping.")

### 21b. Combined — dipole angle vs H (hour) + COSMOS theory

In [ ]:
if "parallactic_angle_deg" in df_clean.columns:
    figure_combined_profile_theory(
        df_clean,
        "hour_angle_hr",
        "r:dipoleAngle",
        np.linspace(-6, 6, 61),
        COSMOS_RA_DEG,
        COSMOS_DEC_DEG,
        suptitle="Combined COSMOS — dipole angle vs H (hour), all selected diaObjects",
        figname="combined_profile_theory_parallactic_vs_Hhr_COSMOS",
        xlabel="Hour angle H (hour)",
        ylabel="r:dipoleAngle (deg, CCW from East)",
        xlim=(-6, 6),
        ylim=(-180, 180),
        x_is_hours=True,
    )
else:
    print("No geometry — skipping.")

### 21c. Combined — rose diagram `r:dipoleAngle`

In [ ]:
figure_combined_rose(
    df_clean,
    "r:dipoleAngle",
    suptitle="Combined COSMOS — rose diagram r:dipoleAngle (all selected objects)",
    figname="combined_rose_dipoleAngle_COSMOS",
) if not df_clean.empty else print("No data.")

### 21d. Combined — dipole length vs tan(z) + DCR prediction per band

In [ ]:
if "tan_zenith" in df_clean.columns and SEP_COL in df_clean.columns:
    mask = df_clean["tan_zenith"].notna() & df_clean[SEP_COL].notna()
    df_p = df_clean[mask]
    bins_tz = np.linspace(0.0, 2.0, 41)
    bands = [b for b in BAND_ORDER if b in df_p["r:band"].dropna().unique()]
    fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
    for band in bands:
        ib = df_p["r:band"] == band
        ax.scatter(
            df_p.loc[ib, "tan_zenith"].values,
            df_p.loc[ib, SEP_COL].values,
            s=5,
            alpha=0.15,
            color=BAND_COLORS.get(band, "grey"),
            rasterized=True,
        )
    for band in bands:
        ib = df_p["r:band"] == band
        xv = df_p.loc[ib, "tan_zenith"].values
        yv = df_p.loc[ib, SEP_COL].values
        xc, ym, ye, _ = profile_median(xv, yv, bins_tz, min_count=3)
        valid = np.isfinite(ym)
        if valid.sum() < 2:
            continue
        color = BAND_COLORS.get(band, "grey")
        ax.errorbar(
            xc[valid],
            ym[valid],
            yerr=ye[valid],
            fmt="o",
            capsize=3,
            color=color,
            ms=5,
            markeredgecolor="black",
            markeredgewidth=0.4,
            elinewidth=1.2,
            label=band,
        )
        tz_line = np.linspace(0.0, 2.0, 200)
        ax.plot(tz_line, SIGMA_N[band] * tz_line * RAD_TO_ARCSEC, ls="--", lw=1.0, color=color, alpha=0.7)
    r_s, p_s = stats.spearmanr(df_p["tan_zenith"].values, df_p[SEP_COL].values)
    ax.text(
        0.02,
        0.97,
        f"Spearman ρ = {r_s:.3f}  (p = {p_s:.2e})   n = {len(df_p):,}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8),
    )
    ax.set_xlim(0.0, 2.0)
    ax.set_ylim(0.0, 0.3)
    ax.set_xlabel("tan(zenith angle)", fontsize=10)
    ax.set_ylabel("r:dipoleLength (arcsec)", fontsize=10)
    ax.set_title("Combined COSMOS — r:dipoleLength vs tan(z)  (dashed = DCR prediction)", fontsize=11)
    ax.legend(fontsize=8, loc="upper left", framealpha=0.8, ncol=2)
    savefig("combined_scatter_dipoleLength_vs_tanz_COSMOS")
    plt.show()
else:
    print("Missing columns — skipping.")

## 22. Same combined figures — after separation cut (`r:dipoleLength` < SEPCUT)

In [ ]:
print(
    f"Combined after cut sep < {SEPCUT}: "
    f"{len(df_selected):,} / {len(df_clean):,} ({100 * len(df_selected) / max(len(df_clean), 1):.1f}%)"
)

### 22a. Combined (cut) — dipole angle vs H (deg)

In [ ]:
if not df_selected.empty and "parallactic_angle_deg" in df_selected.columns:
    figure_combined_profile_theory(
        df_selected,
        "hour_angle_deg",
        "r:dipoleAngle",
        np.linspace(-120, 120, 61),
        COSMOS_RA_DEG,
        COSMOS_DEC_DEG,
        suptitle=f"Combined COSMOS — dipole angle vs H (deg)  (sep < {SEPCUT})",
        figname="combined_profile_theory_parallactic_vs_Hdeg_COSMOS_selected",
        xlabel="Hour angle H (deg)",
        ylabel="r:dipoleAngle (deg, CCW from East)",
        xlim=(-120, 120),
        ylim=(-180, 180),
        x_is_hours=False,
    )

### 22b. Combined (cut) — dipole angle vs H (hour)

In [ ]:
if not df_selected.empty and "parallactic_angle_deg" in df_selected.columns:
    figure_combined_profile_theory(
        df_selected,
        "hour_angle_hr",
        "r:dipoleAngle",
        np.linspace(-6, 6, 61),
        COSMOS_RA_DEG,
        COSMOS_DEC_DEG,
        suptitle=f"Combined COSMOS — dipole angle vs H (hour)  (sep < {SEPCUT})",
        figname="combined_profile_theory_parallactic_vs_Hhr_COSMOS_selected",
        xlabel="Hour angle H (hour)",
        ylabel="r:dipoleAngle (deg, CCW from East)",
        xlim=(-6, 6),
        ylim=(-180, 180),
        x_is_hours=True,
    )

### 22c. Combined (cut) — rose diagram

In [ ]:
figure_combined_rose(
    df_selected,
    "r:dipoleAngle",
    suptitle=f"Combined COSMOS — rose diagram r:dipoleAngle  (sep < {SEPCUT})",
    figname="combined_rose_dipoleAngle_COSMOS_selected",
) if not df_selected.empty else print("No data after cut.")

### 22d. Combined (cut) — dipole length vs tan(z)

In [ ]:
if not df_selected.empty and "tan_zenith" in df_selected.columns:
    mask = df_selected["tan_zenith"].notna() & df_selected[SEP_COL].notna()
    df_ps = df_selected[mask]
    bins_tz = np.linspace(0.0, 2.0, 41)
    bands = [b for b in BAND_ORDER if b in df_ps["r:band"].dropna().unique()]
    fig, ax = plt.subplots(figsize=(9, 5), layout="constrained")
    for band in bands:
        ib = df_ps["r:band"] == band
        ax.scatter(
            df_ps.loc[ib, "tan_zenith"].values,
            df_ps.loc[ib, SEP_COL].values,
            s=5,
            alpha=0.15,
            color=BAND_COLORS.get(band, "grey"),
            rasterized=True,
        )
    for band in bands:
        ib = df_ps["r:band"] == band
        xv = df_ps.loc[ib, "tan_zenith"].values
        yv = df_ps.loc[ib, SEP_COL].values
        xc, ym, ye, _ = profile_median(xv, yv, bins_tz, min_count=3)
        valid = np.isfinite(ym)
        if valid.sum() < 2:
            continue
        color = BAND_COLORS.get(band, "grey")
        ax.errorbar(
            xc[valid],
            ym[valid],
            yerr=ye[valid],
            fmt="o",
            capsize=3,
            color=color,
            ms=5,
            markeredgecolor="black",
            markeredgewidth=0.4,
            elinewidth=1.2,
            label=band,
        )
        tz_line = np.linspace(0.0, 2.0, 200)
        ax.plot(tz_line, SIGMA_N[band] * tz_line * RAD_TO_ARCSEC, ls="--", lw=1.0, color=color, alpha=0.7)
    r_s, p_s = stats.spearmanr(df_ps["tan_zenith"].values, df_ps[SEP_COL].values)
    ax.text(
        0.02,
        0.97,
        f"Spearman ρ = {r_s:.3f}  (p = {p_s:.2e})   n = {len(df_ps):,}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8,
        bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8),
    )
    ax.set_xlim(0.0, 2.0)
    ax.set_ylim(0.0, SEPCUT)
    ax.set_xlabel("tan(zenith angle)", fontsize=10)
    ax.set_ylabel("r:dipoleLength (arcsec)", fontsize=10)
    ax.set_title(f"Combined COSMOS — r:dipoleLength vs tan(z)  (sep < {SEPCUT}, dashed=DCR)", fontsize=11)
    ax.legend(fontsize=8, loc="upper left", framealpha=0.8, ncol=2)
    savefig("combined_scatter_dipoleLength_vs_tanz_COSMOS_selected")
    plt.show()
else:
    print("No data after cut or missing column — skipping.")

## 23. Physical transit zenith distance per object + sigma_n reminder

In [ ]:
print(f"{'diaObjectId':22s}  field  δ (deg)  z_transit (deg)  sin(z_transit)")
print("-" * 72)
for oid in OBJECT_IDS_PRESENT:
    info = OBJECT_INFO[oid]
    dec = info["dec"]
    z_tran = abs(RUBIN_LAT_DEG - dec)
    sinz = np.sin(np.radians(z_tran))
    print(f"  {oid}  {info['field']:7s}  {dec:+7.3f}  {z_tran:15.2f}  {sinz:.4f}")
print()
print(f"{'Band':>5}   {'sigma_n':>12}   {'l_dip(z=45°) [arcsec]':>22}")
print("-" * 46)
for b in BAND_ORDER:
    sn = SIGMA_N[b]
    ldip45 = sn * np.tan(np.deg2rad(45.0)) * RAD_TO_ARCSEC
    print(f"  {b}      {sn:.4e}          {ldip45:.5f}")